# Smartphone Data Simulator

Simulates smartphone sales and saves them as `smartphone_data.csv`, which `smartphone_prices.ipynb` uses for the boxplots and the ANOVA.

[Faker](https://faker.readthedocs.io) is a Python library that generates realistic fake data such as names, addresses, cities, dates or IDs. It supports many locales (e.g. `de_CH` for Switzerland) and lets you add your own *providers* for domain-specific values such as smartphone brands and models.

In this example, Faker creates the categorical and descriptive columns (brand, model, shop city, date, payment method), while NumPy simulates the prices from statistical distributions.

Install it once with `pip install faker`.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
from collections import OrderedDict
from faker import Faker
from faker.providers import DynamicProvider
from IPython.display import Image

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

In [ ]:
Image("smartphones.jpg", width=500)

## Random seed
Seed both Faker and NumPy, so the simulated data is reproducible. Use your own birthday, so your data is unique to you.

In [ ]:
day, month = 15, 6  # <-- replace with your own birth day and month
seed = day + month

fake = Faker('de_CH')          # Swiss locale: Swiss cities, names, ...
Faker.seed(seed)               # seed for Faker
rng = np.random.default_rng(seed)  # seed for NumPy

## Domain knowledge: brands, models and prices
Faker knows nothing about smartphones, so we describe them ourselves. The list prices (CHF) are approximate and for illustration only.

In [ ]:
# Approximate list prices in CHF for the 128 GB version of each model
catalog = {
    'Apple':   {'iPhone 16': 899, 'iPhone 16 Pro': 1199, 'iPhone 15': 749},
    'Samsung': {'Galaxy S24': 849, 'Galaxy S24 Ultra': 1299, 'Galaxy A55': 449},
    'Google':  {'Pixel 9': 799, 'Pixel 9 Pro': 1099, 'Pixel 8a': 499},
}

# Extra charge for more storage
storage_surcharge = {128: 0, 256: 100, 512: 300}

# Custom Faker provider: fake.smartphone_brand() returns one of the brands
brand_provider = DynamicProvider(provider_name='smartphone_brand', elements=list(catalog))
fake.add_provider(brand_provider)

print(fake.smartphone_brand(), fake.smartphone_brand(), fake.smartphone_brand())

## Simulate sales
Each row is one smartphone sale. Faker draws the brand, model, storage, shop city, date and payment method (`random_element` with weights for unequal shares). NumPy draws a discount, so prices vary around the list price.

In [ ]:
n = 1000  # number of simulated sales

data = []
for _ in range(n):
    brand = fake.smartphone_brand()
    model = fake.random_element(list(catalog[brand]))
    storage_gb = fake.random_element(OrderedDict([(128, 0.5), (256, 0.35), (512, 0.15)]))
    list_price = catalog[brand][model] + storage_surcharge[storage_gb]
    discount = np.clip(rng.normal(loc=0.05, scale=0.04), 0, 0.25)  # 0-25 % discount
    data.append({
        'sale_id': fake.bothify('S-####-??').upper(),
        'date': fake.date_between(start_date='-1y', end_date='today'),
        'shop_city': fake.city(),
        'brand': brand,
        'model': model,
        'storage_gb': storage_gb,
        'price_chf': round(list_price * (1 - discount)),
        'payment': fake.random_element(OrderedDict([('Card', 0.6), ('TWINT', 0.3), ('Cash', 0.1)])),
    })

df = pd.DataFrame(data)
df.head(10)

In [ ]:
# Overview
print(df.shape)
print(df['brand'].value_counts())
df.groupby('brand')['price_chf'].describe().round(0)

## Save the simulated data
The other notebooks of this exercise read the data from this file.

In [ ]:
df.to_csv('smartphone_data.csv', index=False)
print('Saved', len(df), 'rows to smartphone_data.csv')

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')